# LeetCode 2294 — Partition Array Such That Maximum Difference Is K bằng Python

Link đề: https://leetcode.com/problems/partition-array-such-that-maximum-difference-is-k/ · Độ khó: **Medium** · Chủ đề: Array, Greedy, Sorting

Skeleton LeetCode:

```python
class Solution:
    def partitionArray(self, nums: List[int], k: int) -> int:
        ...
```

## 1. Mục tiêu học

1. Nhận ra rằng chia thành các **dãy con** (subsequence, không cần liên tiếp) nghĩa là **thứ tự ban đầu không quan trọng** ⇒ được phép **sắp xếp**.
2. Chứng minh: luôn tồn tại lời giải tối ưu mà mỗi nhóm là một **đoạn liên tiếp** trong mảng đã sắp xếp.
3. Tham lam "mở nhóm mới khi `x − min_nhóm > k`" và chứng minh bằng lập luận **luôn đi trước** (greedy stays ahead) / bằng tập "điểm phủ".
4. So sánh với DP O(n²) trên mảng đã sắp xếp, với cách nhảy bằng `bisect` O(g log n), và với **đếm phân phối** O(n + max) khi giá trị nhỏ.
5. Nhận diện các bẫy: so với phần tử **trước** thay vì phần tử **đầu nhóm**, quên sắp xếp, nhầm `>` với `≥`.

## 2. Đề bài & ví dụ

Cho mảng `nums` và số nguyên `k`. Chia `nums` thành một hoặc nhiều **dãy con** sao cho mỗi phần tử thuộc **đúng một** dãy con. Trả về **số dãy con ít nhất** sao cho trong mỗi dãy con, `max − min ≤ k`.

**Ví dụ 1:** `nums = [3,6,1,2,5]`, `k = 2` → `2` (`[3,1,2]` và `[6,5]`).

**Ví dụ 2:** `nums = [1,2,3]`, `k = 1` → `2` (`[1,2]` và `[3]`, hoặc `[1]` và `[2,3]`).

**Ví dụ 3:** `nums = [2,2,4,5]`, `k = 0` → `3` (`[2,2]`, `[4]`, `[5]`).

**Ràng buộc:** `1 ≤ n ≤ 10⁵`; `0 ≤ nums[i] ≤ 10⁵`; `0 ≤ k ≤ 10⁵`.

In [1]:
from typing import List
from bisect import bisect_right
from functools import lru_cache
import random, time

examples = [([3, 6, 1, 2, 5], 2, 2), ([1, 2, 3], 1, 2), ([2, 2, 4, 5], 0, 3)]
for a, k, exp in examples:
    print(f"nums={a} k={k} -> kỳ vọng {exp}")

nums=[3, 6, 1, 2, 5] k=2 -> kỳ vọng 2
nums=[1, 2, 3] k=1 -> kỳ vọng 2
nums=[2, 2, 4, 5] k=0 -> kỳ vọng 3


## 3. Kiến thức nền

### 3.1 Dãy con vs đoạn con

Dãy con giữ **thứ tự** nhưng không cần liên tiếp. Điều kiện `max − min ≤ k` của một nhóm chỉ phụ thuộc vào **tập giá trị** của nhóm, không phụ thuộc thứ tự ⇒ ta có thể coi `nums` là một **đa tập** (multiset) và sắp xếp tùy ý. Mọi cách chia tập chỉ số thành các nhóm đều tương ứng với một cách chia thành dãy con (lấy các phần tử theo thứ tự chỉ số).

In [2]:
a = [3, 6, 1, 2, 5]
groups_idx = [[0, 2, 3], [1, 4]]
print("nhóm theo chỉ số:", groups_idx, "-> dãy con:", [[a[i] for i in g] for g in groups_idx])
print("max-min mỗi nhóm:", [max(a[i] for i in g) - min(a[i] for i in g) for g in groups_idx])

nhóm theo chỉ số: [[0, 2, 3], [1, 4]] -> dãy con: [[3, 1, 2], [6, 5]]
max-min mỗi nhóm: [2, 1]


### 3.2 Sắp xếp và "nhóm là đoạn liên tiếp trong mảng đã sắp xếp"

Nếu một nhóm chứa `x` và `z` (`x ≤ z`) thì mọi `y` với `x ≤ y ≤ z` đều có thể thêm vào nhóm mà không làm hỏng điều kiện (max và min không đổi). Đây là gợi ý rằng nhóm nên là **khoảng giá trị** — mục 10 chứng minh chặt chẽ.

### 3.3 `bisect_right`: vị trí đầu tiên có giá trị `> t` trong mảng đã sắp xếp

In [3]:
s = sorted([3, 6, 1, 2, 5])
print(s, "| bisect_right(s, 1 + 2) =", bisect_right(s, 3), "-> nhóm đầu là", s[:bisect_right(s, 3)])

[1, 2, 3, 5, 6] | bisect_right(s, 1 + 2) = 3 -> nhóm đầu là [1, 2, 3]


### 3.4 Đếm phân phối (giá trị ≤ 10⁵)

Chỉ cần biết **giá trị nào có mặt** (số lần lặp không ảnh hưởng số nhóm) ⇒ mảng đánh dấu `present[v]`.

In [4]:
vals = [2, 2, 4, 5]
present = [False] * 6
for v in vals: present[v] = True
print([v for v in range(6) if present[v]])

[2, 4, 5]


### 3.5 Từ nhóm giá trị về **dãy con thật** trong mảng gốc

Đề chỉ hỏi số nhóm, nhưng để thấy mọi thứ khớp với định nghĩa "dãy con", ta gán **nhãn nhóm** cho từng chỉ số rồi gom theo thứ tự chỉ số. Sắp xếp **chỉ số** theo giá trị (`sorted(range(n), key=nums.__getitem__)`) giữ được liên hệ với vị trí gốc.

In [5]:
def subsequences(nums, k):
    order = sorted(range(len(nums)), key=nums.__getitem__)
    label = [0] * len(nums); g = 0; start = nums[order[0]]
    for i in order:
        if nums[i] - start > k: g += 1; start = nums[i]
        label[i] = g
    out = [[] for _ in range(g + 1)]
    for i, x in enumerate(nums): out[label[i]].append(x)      # duyệt theo chỉ số => giữ thứ tự gốc
    return out

for a, k, _ in examples:
    print(a, "k =", k, "->", subsequences(a, k))

[3, 6, 1, 2, 5] k = 2 -> [[3, 1, 2], [6, 5]]
[1, 2, 3] k = 1 -> [[1, 2], [3]]
[2, 2, 4, 5] k = 0 -> [[2, 2], [4], [5]]


## 4. Ý tưởng nền tảng / trực giác

Sắp xếp `nums` tăng dần. Phần tử **nhỏ nhất** `s[0]` phải thuộc một nhóm nào đó; nhóm đó chỉ chứa được các giá trị trong `[s[0], s[0] + k]`. Không có lý do gì để **không** nhét tất cả các giá trị trong khoảng này vào nhóm đó — chúng chỉ có thể làm các nhóm khác "dễ thở" hơn. Rồi lặp lại với phần tử nhỏ nhất còn lại.

Thuật toán: duyệt mảng đã sắp, giữ `start` = giá trị nhỏ nhất của nhóm hiện tại; **nếu** `x − start > k` **thì** mở nhóm mới với `start = x`.

Tính tay ví dụ 1: `s = [1, 2, 3, 5, 6]`, `k = 2`.

| x | x − start | hành động | start | số nhóm |
|---|---|---|---|---|
| 1 | 0 | bắt đầu nhóm 1 | 1 | 1 |
| 2 | 1 | vào nhóm | 1 | 1 |
| 3 | 2 | vào nhóm (2 ≤ k) | 1 | 1 |
| 5 | 4 | **mở nhóm mới** | 5 | 2 |
| 6 | 1 | vào nhóm | 5 | 2 |

⇒ 2 ✔. Ví dụ 3, `k = 0`: `[2, 2, 4, 5]` ⇒ nhóm `{2, 2}`, `{4}`, `{5}` ⇒ 3 ✔.

In [6]:
def greedy_groups(nums, k):
    s = sorted(nums); groups = [[s[0]]]
    for x in s[1:]:
        if x - groups[-1][0] > k: groups.append([x])
        else: groups[-1].append(x)
    return groups

for a, k, exp in examples:
    g = greedy_groups(a, k); print(g, "->", len(g), "| kỳ vọng", exp)

[[1, 2, 3], [5, 6]] -> 2 | kỳ vọng 2
[[1, 2], [3]] -> 2 | kỳ vọng 2
[[2, 2], [4], [5]] -> 3 | kỳ vọng 3


## 5. Cách 1 — Brute Force / baseline

### 5.1 Vét cạn mọi cách chia tập (số Bell) — chỉ cho n ≤ 9

Gán lần lượt từng phần tử vào một nhóm đã có hoặc một nhóm mới, cắt nhánh khi số nhóm ≥ kết quả tốt nhất. Số cách chia tập `n` phần tử là số Bell `B(n)` (`B(9) = 21147`).

In [7]:
def brute(nums, k):
    n = len(nums); best = [n]
    groups = []                                  # mỗi nhóm lưu (min, max)
    def go(i):
        if len(groups) >= best[0]: return
        if i == n: best[0] = len(groups); return
        x = nums[i]
        for g in range(len(groups)):
            lo, hi = groups[g]
            if max(hi, x) - min(lo, x) <= k:
                groups[g] = (min(lo, x), max(hi, x)); go(i + 1); groups[g] = (lo, hi)
        groups.append((x, x)); go(i + 1); groups.pop()
    go(0)
    return best[0]

for a, k, exp in examples:
    print(brute(a, k), "OK" if brute(a, k) == exp else "SAI")

2 OK
2 OK
3 OK


**Độ phức tạp:** cỡ `B(n)` — siêu mũ. Không dùng được cho `n = 10⁵`, chỉ để kiểm chứng.

### 5.2 Baseline 2 — DP trên mảng đã sắp xếp — O(n²)

Nếu chấp nhận (chứng minh ở mục 10) rằng các nhóm là đoạn liên tiếp của mảng đã sắp, thì `dp[i]` = số nhóm ít nhất cho `i` phần tử nhỏ nhất: `dp[i] = min(dp[j] + 1 : s[i−1] − s[j] ≤ k)`.

In [8]:
def dp_quadratic(nums, k):
    s = sorted(nums); n = len(s)
    dp = [0] + [n + 1] * n
    for i in range(1, n + 1):
        for j in range(i):                       # nhóm cuối là s[j..i-1]
            if s[i - 1] - s[j] <= k:
                dp[i] = min(dp[i], dp[j] + 1)
    return dp[n]

for a, k, exp in examples:
    print(dp_quadratic(a, k), "OK" if dp_quadratic(a, k) == exp else "SAI")

2 OK
2 OK
3 OK


**Nút thắt:** O(n²) = `10¹⁰`. Quan sát: `dp` **không giảm** theo `i` (bỏ bớt phần tử lớn nhất không làm tăng số nhóm), nên trong các `j` hợp lệ, `j` **nhỏ nhất** luôn tốt nhất ⇒ chỉ cần một con trỏ — và đó chính là tham lam.

## 6. Ý tưởng thuật toán / bước đột phá

**Sắp xếp + quét tham lam.**

```
sort(nums); ans = 1; start = nums[0]
for x in nums:
    if x - start > k: ans += 1; start = x
return ans
```

**Cách nhìn thứ hai — "điểm phủ" (interval stabbing):** coi mỗi nhóm là một **đoạn giá trị độ dài `k`** `[a, a + k]`; mỗi giá trị phải nằm trong một đoạn. Bài toán = phủ các điểm trên trục số bằng ít đoạn độ dài `k` nhất. Tham lam kinh điển: đặt đoạn bắt đầu tại điểm **chưa phủ nhỏ nhất**.

**Độ phức tạp:** O(n log n) cho sắp xếp + O(n) quét. Với giá trị ≤ 10⁵ có thể dùng đếm phân phối O(n + V).

### 6.1 Hình dung "phủ điểm bằng đoạn độ dài k" trên trục số

Mỗi dòng là một nhóm (một đoạn `[start, start + k]`); dấu `o` là giá trị có mặt, `[` `]` là hai đầu đoạn. Các đoạn không chồng lên nhau ở **điểm có mặt** — mỗi điểm thuộc đúng một đoạn.

In [9]:
def draw(nums, k):
    V = max(nums) + k + 1; pres = set(nums)
    print("     " + "".join(str(v % 10) for v in range(V)))
    print("điểm " + "".join("o" if v in pres else "." for v in range(V)))
    for t, g in enumerate(greedy_groups(nums, k), 1):
        a = g[0]; line = [" "] * V
        for v in range(a, a + k + 1): line[v] = "-"
        line[a] = "["; line[a + k] = "]"
        print(f"n{t:<3} " + "".join(line))

draw([10, 1, 4, 4, 7, 2, 13, 11], 3)

     01234567890123456
điểm .oo.o..o..oo.o...
n1    [--]            
n2          [--]      
n3              [--]  


Nhìn hình: đoạn thứ hai bắt đầu tại 7 (điểm chưa phủ nhỏ nhất sau đoạn `[1, 4]`) chứ không tại 5 — vì 5, 6 không có mặt; đặt đoạn tại điểm có mặt là cách "đẩy" đoạn sang phải xa nhất có thể mà vẫn phủ được điểm nhỏ nhất chưa phủ. Đó là lý do tham lam tối ưu.

## 7. Các cách tiếp theo

### 7.1 Cách 2 — Sắp xếp + quét (cách doocs)

In [10]:
def sort_scan(nums, k):
    s = sorted(nums)
    ans, a = 1, s[0]
    for b in s:
        if b - a > k:
            a = b; ans += 1
    return ans

for a, k, exp in examples:
    print(sort_scan(a, k), "OK" if sort_scan(a, k) == exp else "SAI")

2 OK
2 OK
3 OK


### 7.2 Cách 3 — Nhảy bằng `bisect`: O(n log n + g log n) với `g` là số nhóm

Sau khi sắp xếp, nhóm bắt đầu tại chỉ số `i` kết thúc ngay trước `bisect_right(s, s[i] + k)`. Nhảy thẳng tới đó. Có lợi khi `g` nhỏ (k lớn).

In [11]:
def bisect_jump(nums, k):
    s = sorted(nums); i = 0; ans = 0
    while i < len(s):
        ans += 1
        i = bisect_right(s, s[i] + k, i)
    return ans

for a, k, exp in examples:
    print(bisect_jump(a, k), "OK" if bisect_jump(a, k) == exp else "SAI")

2 OK
2 OK
3 OK


### 7.3 Cách 4 — Đếm phân phối: O(n + V), không sắp xếp so sánh

Đánh dấu các giá trị có mặt, rồi quét `v = 0 … V`: gặp giá trị có mặt mà chưa được phủ ⇒ mở nhóm phủ `[v, v + k]` và nhảy `v` tới `v + k + 1`.

In [12]:
def counting(nums, k):
    V = max(nums)
    present = bytearray(V + 1)
    for x in nums: present[x] = 1
    ans = 0; v = 0
    while v <= V:
        if present[v]:
            ans += 1; v += k + 1                 # mọi giá trị trong [v, v+k] đã được phủ
        else:
            v += 1
    return ans

for a, k, exp in examples:
    print(counting(a, k), "OK" if counting(a, k) == exp else "SAI")

2 OK
2 OK
3 OK


### 7.4 Cách 5 — Tham lam từ phía **lớn nhất** (đối xứng)

Phủ điểm lớn nhất chưa phủ bằng đoạn `[x − k, x]`. Kết quả giống hệt — minh họa rằng tham lam đúng theo cả hai chiều (nhóm có thể khác nhau, nhưng số nhóm như nhau).

In [13]:
def from_top(nums, k):
    s = sorted(nums, reverse=True); ans, top = 1, s[0]
    for x in s:
        if top - x > k: top = x; ans += 1
    return ans

for a, k, exp in examples:
    print(from_top(a, k), "OK" if from_top(a, k) == exp else "SAI")

2 OK
2 OK
3 OK


## 8. TRACE từng bước

### 8.1 Trace quét tham lam trên một mảng có lặp và khoảng trống

In [14]:
def trace(nums, k):
    s = sorted(nums); ans, start = 1, s[0]
    print("sắp xếp:", s, "k =", k)
    for x in s:
        if x - start > k:
            ans += 1; print(f"  x={x:<3} x-start={x - start:<3} > k  -> mở nhóm {ans}, start = {x}"); start = x
        else:
            print(f"  x={x:<3} x-start={x - start:<3} <= k -> vào nhóm {ans}")
    return ans

print("đáp án:", trace([10, 1, 4, 4, 7, 2, 13, 11], 3), "| brute:", brute([10, 1, 4, 4, 7, 2, 13, 11], 3))

sắp xếp: [1, 2, 4, 4, 7, 10, 11, 13] k = 3
  x=1   x-start=0   <= k -> vào nhóm 1
  x=2   x-start=1   <= k -> vào nhóm 1
  x=4   x-start=3   <= k -> vào nhóm 1
  x=4   x-start=3   <= k -> vào nhóm 1
  x=7   x-start=6   > k  -> mở nhóm 2, start = 7
  x=10  x-start=3   <= k -> vào nhóm 2
  x=11  x-start=4   > k  -> mở nhóm 3, start = 11
  x=13  x-start=2   <= k -> vào nhóm 3
đáp án: 3 | brute: 3


Kết quả: nhóm 1 = `{1, 2, 4, 4}` (start 1, phủ tới 4); `7 − 1 = 6 > 3` ⇒ nhóm 2 bắt đầu ở 7, gồm `{7, 10}`; `11 − 7 = 4 > 3` ⇒ nhóm 3 = `{11, 13}`. Tổng 3 nhóm, khớp vét cạn.

### 8.2 Trace cách nhảy `bisect`

In [15]:
s = sorted([10, 1, 4, 4, 7, 2, 13, 11]); k = 3; i = 0; g = 0
while i < len(s):
    j = bisect_right(s, s[i] + k, i); g += 1
    print(f"nhóm {g}: bắt đầu s[{i}]={s[i]}, phủ tới giá trị {s[i] + k}, gồm {s[i:j]}, nhảy tới chỉ số {j}")
    i = j

nhóm 1: bắt đầu s[0]=1, phủ tới giá trị 4, gồm [1, 2, 4, 4], nhảy tới chỉ số 4
nhóm 2: bắt đầu s[4]=7, phủ tới giá trị 10, gồm [7, 10], nhảy tới chỉ số 6
nhóm 3: bắt đầu s[6]=11, phủ tới giá trị 14, gồm [11, 13], nhảy tới chỉ số 8


## 9. Code nộp LeetCode

In [16]:
from typing import List

class Solution:
    def partitionArray(self, nums: List[int], k: int) -> int:
        nums = sorted(nums)                     # dãy con: thứ tự không quan trọng
        ans, start = 1, nums[0]                 # start = min của nhóm hiện tại
        for x in nums:
            if x - start > k:                   # x không thể chung nhóm với start
                ans += 1
                start = x
        return ans

In [17]:
sol = Solution()
for a, k, exp in examples:
    got = sol.partitionArray(a, k)
    print(got, "| kỳ vọng", exp, "| OK" if got == exp else "| SAI")

2 | kỳ vọng 2 | OK
2 | kỳ vọng 2 | OK
3 | kỳ vọng 3 | OK


## 10. Vì sao thuật toán đúng

Gọi `s₁ ≤ s₂ ≤ … ≤ sₙ` là mảng đã sắp. Tham lam chọn các **điểm bắt đầu** `a₁ = s₁ < a₂ < … < a_g`, với `a_{t+1}` = phần tử đầu tiên `> a_t + k`.

**Chặn dưới (mọi cách chia cần ≥ g nhóm).** Xét các phần tử `a₁, …, a_g`. Với `t < u`: `a_u ≥ a_{t+1} > a_t + k` ⇒ `a_u − a_t > k` ⇒ **không hai điểm bắt đầu nào có thể chung nhóm**. Vậy mọi cách chia hợp lệ có ít nhất `g` nhóm.

**Chặn trên (tham lam hợp lệ).** Nhóm `t` của tham lam gồm các phần tử trong `[a_t, a_t + k]` (từ `a_t` đến ngay trước `a_{t+1}`) ⇒ `max − min ≤ k`. Mọi phần tử thuộc đúng một nhóm. Vậy tham lam dùng đúng `g` nhóm hợp lệ.

⇒ Tham lam tối ưu, và đáp án = `g`. ∎

**Bổ sung — vì sao được sắp xếp?** Điều kiện chỉ phụ thuộc vào tập giá trị của từng nhóm; một cách chia tập chỉ số luôn cho ra các dãy con hợp lệ (lấy phần tử theo thứ tự chỉ số). Vậy bài toán trên mảng gốc tương đương bài toán trên đa tập.

**Bổ sung — chứng minh `dp` không giảm (mục 5.2).** Bỏ phần tử lớn nhất khỏi một cách chia hợp lệ của `i` phần tử nhỏ nhất cho ra một cách chia hợp lệ của `i − 1` phần tử với không nhiều nhóm hơn ⇒ `dp[i−1] ≤ dp[i]`.

In [18]:
# minh họa chặn dưới: các điểm bắt đầu đôi một cách nhau > k
rng = random.Random(10)
for _ in range(500):
    a = [rng.randint(0, 30) for _ in range(rng.randint(1, 12))]; k = rng.randint(0, 6)
    starts = [g[0] for g in greedy_groups(a, k)]
    assert all(y - x > k for x, y in zip(starts, starts[1:]))
    assert all(g[-1] - g[0] <= k for g in greedy_groups(a, k))
print("điểm bắt đầu:", [g[0] for g in greedy_groups([10, 1, 4, 4, 7, 2, 13, 11], 3)], "- đôi một cách nhau > 3")

điểm bắt đầu: [1, 7, 11] - đôi một cách nhau > 3


## 11. Phân tích Time / Space Complexity

`n ≤ 10⁵`, `V = max(nums) ≤ 10⁵`, `g` = số nhóm.

| Cách | Time | Space |
|---|---|---|
| 1. Vét cạn chia tập | ~B(n) | O(n) |
| 1b. DP trên mảng sắp | O(n²) | O(n) |
| 2. Sắp xếp + quét | O(n log n) | O(n) (bản sao đã sắp) |
| 3. `bisect` nhảy | O(n log n + g log n) | O(n) |
| 4. Đếm phân phối | O(n + V) | O(V) |
| 5. Từ phía lớn | O(n log n) | O(n) |

In [19]:
rng = random.Random(2294)
big = [rng.randint(0, 10**5) for _ in range(10**5)]
for k in [0, 10, 1000, 10**5]:
    res = []
    for name, f in [("sort_scan", sort_scan), ("bisect_jump", bisect_jump), ("counting", counting),
                    ("from_top", from_top), ("Solution", Solution().partitionArray)]:
        t0 = time.perf_counter(); res.append(f(big, k)); dt = time.perf_counter() - t0
        print(f"k={k:<6} {name:<11} {dt*1000:6.1f} ms -> {res[-1]}")
    assert len(set(res)) == 1

k=0      sort_scan     19.2 ms -> 63173
k=0      bisect_jump   27.2 ms -> 63173
k=0      counting       7.2 ms -> 63173
k=0      from_top      18.4 ms -> 63173


k=0      Solution      18.7 ms -> 63173


k=10     sort_scan     18.5 ms -> 8623
k=10     bisect_jump   16.8 ms -> 8623
k=10     counting       2.7 ms -> 8623
k=10     from_top      18.8 ms -> 8623
k=10     Solution      21.5 ms -> 8623


k=1000   sort_scan     17.1 ms -> 100
k=1000   bisect_jump   14.3 ms -> 100
k=1000   counting       2.0 ms -> 100
k=1000   from_top      16.9 ms -> 100


k=1000   Solution      17.8 ms -> 100
k=100000 sort_scan     16.6 ms -> 1


k=100000 bisect_jump   14.8 ms -> 1
k=100000 counting       2.0 ms -> 1
k=100000 from_top      17.0 ms -> 1
k=100000 Solution      21.2 ms -> 1


## 12. Bộ test

### 12.1 Ví dụ đề + edge case

In [20]:
ALL = {"brute": brute, "dp_quadratic": dp_quadratic, "sort_scan": sort_scan, "bisect_jump": bisect_jump,
       "counting": counting, "from_top": from_top, "Solution": Solution().partitionArray}
edge = examples + [
    ([5], 0, 1),
    ([0], 100000, 1),
    ([7, 7, 7, 7], 0, 1),                    # trùng hết, k = 0
    ([1, 2, 3, 4, 5], 0, 5),                 # khác hết, k = 0
    ([1, 2, 3, 4, 5], 100, 1),               # k rất lớn
    ([1, 3, 5, 7], 2, 2),                    # đúng biên: 3-1 = 2 <= k
    ([1, 4, 7, 10], 2, 4),                   # cách nhau đúng k+1
    ([0, 100000], 99999, 2),
    ([0, 100000], 100000, 1),
]
for a, k, exp in edge:
    for name, f in ALL.items():
        got = f(list(a), k)
        assert got == exp, (name, a, k, got, exp)
    print(f"{str(a):<18} k={k:<6} -> {exp}")
print("edge OK")

[3, 6, 1, 2, 5]    k=2      -> 2
[1, 2, 3]          k=1      -> 2
[2, 2, 4, 5]       k=0      -> 3
[5]                k=0      -> 1
[0]                k=100000 -> 1
[7, 7, 7, 7]       k=0      -> 1
[1, 2, 3, 4, 5]    k=0      -> 5
[1, 2, 3, 4, 5]    k=100    -> 1
[1, 3, 5, 7]       k=2      -> 2
[1, 4, 7, 10]      k=2      -> 4
[0, 100000]        k=99999  -> 2
[0, 100000]        k=100000 -> 1
edge OK


### 12.2 Đối chiếu ngẫu nhiên với vét cạn (seed cố định)

In [21]:
rng = random.Random(42)
for _ in range(1500):
    a = [rng.randint(0, rng.choice([5, 15, 40])) for _ in range(rng.randint(1, 8))]; k = rng.randint(0, 8)
    ref = brute(a, k)
    for name, f in ALL.items():
        assert f(list(a), k) == ref, (name, a, k)
print("1500 bộ ngẫu nhiên OK")

1500 bộ ngẫu nhiên OK


### 12.3 Tính chất: đáp án không tăng khi `k` tăng; hoán vị mảng không đổi đáp án; nhân đôi phần tử không đổi đáp án

In [22]:
rng = random.Random(7)
for _ in range(500):
    a = [rng.randint(0, 50) for _ in range(rng.randint(1, 30))]
    vals = [Solution().partitionArray(a, k) for k in range(0, 52)]
    assert all(x >= y for x, y in zip(vals, vals[1:])) and vals[-1] == 1
    assert vals[0] == len(set(a))
    b = a[:]; rng.shuffle(b)
    assert Solution().partitionArray(b, 5) == vals[5]
    assert Solution().partitionArray(a + a, 5) == vals[5]
print("tính chất OK")

tính chất OK


`k = 0` ⇒ đáp án = **số giá trị phân biệt** (mỗi nhóm chỉ chứa một giá trị). `k ≥ max − min` ⇒ 1 nhóm.

### 12.4 Edge cực trị: n = 10⁵ giá trị cách nhau đúng `k + 1`

In [23]:
k = 0
a = list(range(10**5))
assert Solution().partitionArray(a, 0) == 10**5
a2 = [i * 3 for i in range(33334)]
print(Solution().partitionArray(a2, 2), "(cách nhau 3 với k = 2: mỗi nhóm 1 phần tử -> 33334)")
assert Solution().partitionArray(a2, 2) == 33334 and Solution().partitionArray(a2, 3) == 16667

33334 (cách nhau 3 với k = 2: mỗi nhóm 1 phần tử -> 33334)


### 12.5 Nhóm có thể khác nhau nhưng số nhóm luôn bằng nhau (tham lam từ dưới vs từ trên)

In [24]:
def groups_from_top(nums, k):
    s = sorted(nums, reverse=True); groups = [[s[0]]]
    for x in s[1:]:
        if groups[-1][0] - x > k: groups.append([x])
        else: groups[-1].append(x)
    return [sorted(g) for g in reversed(groups)]

a, k = [1, 2, 3], 1
print("từ dưới:", greedy_groups(a, k), "| từ trên:", groups_from_top(a, k))
rng = random.Random(125); diff = 0
for _ in range(2000):
    a = [rng.randint(0, 20) for _ in range(rng.randint(1, 10))]; k = rng.randint(0, 5)
    g1, g2 = greedy_groups(a, k), groups_from_top(a, k)
    assert len(g1) == len(g2)
    diff += g1 != g2
print("số bộ mà hai cách cho nhóm khác nhau:", diff, "/ 2000 — nhưng số nhóm luôn bằng nhau")

từ dưới: [[1, 2], [3]] | từ trên: [[1], [2, 3]]
số bộ mà hai cách cho nhóm khác nhau: 474 / 2000 — nhưng số nhóm luôn bằng nhau


Ví dụ 2 của đề cũng nói rõ: `[1,2],[3]` và `[1],[2,3]` đều tối ưu. Vì vậy test chỉ nên so **số nhóm**, hoặc kiểm tra tính hợp lệ của nhóm, chứ không so nguyên danh sách nhóm.

## 13. Các lỗi người mới hay gặp

### Lỗi 1 — So với phần tử **liền trước** thay vì phần tử **đầu nhóm**

In [25]:
def wrong_prev(nums, k):
    s = sorted(nums); ans = 1
    for i in range(1, len(s)):
        if s[i] - s[i - 1] > k: ans += 1          # SAI: chuỗi bước nhỏ vẫn làm max - min vượt k
    return ans

for a, k in [([1, 2, 3], 1), ([3, 6, 1, 2, 5], 2)]:
    print("bản sai :", wrong_prev(a, k), "| bản đúng:", Solution().partitionArray(a, k))

bản sai : 1 | bản đúng: 2
bản sai : 1 | bản đúng: 2


**Vì sao sai:** `[1, 2, 3]`, `k = 1`: các bước đều ≤ 1 nên bản sai gộp cả ba, nhưng `3 − 1 = 2 > 1`. Ví dụ 1 cũng sai: mảng sắp `[1, 2, 3, 5, 6]` có các bước `1, 1, 2, 1` đều ≤ 2 nên bản sai trả 1 nhóm, trong khi `6 − 1 = 5`. Ràng buộc là trên **toàn nhóm** (max − min), không phải trên từng cặp kề nhau — các bước nhỏ cộng dồn lại. **Sửa:** so với `start` của nhóm.

### Lỗi 2 — Quên sắp xếp (xử lý như đoạn con liên tiếp)

In [26]:
def wrong_no_sort(nums, k):
    ans, lo, hi = 1, nums[0], nums[0]
    for x in nums:
        lo, hi = min(lo, x), max(hi, x)
        if hi - lo > k: ans += 1; lo = hi = x       # SAI: chỉ chia được thành các đoạn liên tiếp
    return ans

for a, k, _ in examples:
    print("bản sai :", wrong_no_sort(a, k), "| bản đúng:", Solution().partitionArray(a, k))

bản sai : 4 | bản đúng: 2
bản sai : 2 | bản đúng: 2
bản sai : 3 | bản đúng: 3


**Vì sao sai:** ví dụ 1: theo thứ tự gốc `[3, 6, 1, 2, 5]` thì `3, 6` đã vượt `k` ⇒ cắt, rồi `6, 1` cắt, `1, 2` ổn, `2, 5` cắt ⇒ 4 nhóm. Ví dụ 2 và 3 **trùng hợp** đúng vì mảng đã được sắp sẵn. Đề cho phép **dãy con** (bỏ cách quãng) ⇒ phải sắp xếp. Cách này là lời giải đúng của biến thể "đoạn con" (mục 15.2).

### Lỗi 3 — Dùng `≥ k` thay vì `> k`

In [27]:
def wrong_ge(nums, k):
    s = sorted(nums); ans, a = 1, s[0]
    for b in s:
        if b - a >= k: a = b; ans += 1              # SAI: hiệu bằng k vẫn hợp lệ
    return ans

for a, k, _ in examples:
    print("bản sai :", wrong_ge(a, k), "| bản đúng:", Solution().partitionArray(a, k))

bản sai : 3 | bản đúng: 2
bản sai : 3 | bản đúng: 2
bản sai : 5 | bản đúng: 3


**Vì sao sai:** đề cho `max − min ≤ k` (được bằng). Ví dụ 3 (`k = 0`) bản sai còn tách cả hai số `2` (hiệu 0 ≥ 0).

### Lỗi 4 — Công thức "khoảng chia đều": `(max − min) // (k + 1) + 1`

In [28]:
def wrong_formula(nums, k):
    return (max(nums) - min(nums)) // (k + 1) + 1    # SAI: bỏ qua các khoảng trống không có giá trị

for a, k in [([1, 100], 1), ([2, 2, 4, 5], 0)]:
    print("bản sai :", wrong_formula(a, k), "| bản đúng:", Solution().partitionArray(a, k))

bản sai : 50 | bản đúng: 2
bản sai : 4 | bản đúng: 3


**Vì sao sai:** công thức giả định mọi giá trị trong `[min, max]` đều có mặt. `[1, 100]` với `k = 1` chỉ cần 2 nhóm, công thức cho 50.

### Lỗi 5 — Khởi tạo `ans = 0` và chỉ đếm khi mở nhóm mới (quên nhóm đầu)

In [29]:
def wrong_init(nums, k):
    s = sorted(nums); ans, a = 0, s[0]
    for b in s:
        if b - a > k: a = b; ans += 1               # SAI: nhóm chứa s[0] chưa được đếm
    return ans

for a, k, _ in examples:
    print("bản sai :", wrong_init(a, k), "| bản đúng:", Solution().partitionArray(a, k))

bản sai : 1 | bản đúng: 2
bản sai : 1 | bản đúng: 2
bản sai : 2 | bản đúng: 3


**Sửa:** `ans = 1` (mảng không rỗng nên luôn có ít nhất một nhóm), hoặc đếm mỗi lần **bắt đầu** một nhóm kể cả nhóm đầu (như cách `bisect_jump`).

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- "Chia thành **dãy con** / **nhóm** với ràng buộc chỉ phụ thuộc giá trị" ⇒ thứ tự không quan trọng ⇒ **sắp xếp**.
- Ràng buộc `max − min ≤ k` ⇒ nhóm là **khoảng giá trị** ⇒ bài toán **phủ điểm bằng đoạn độ dài k** ⇒ tham lam từ điểm nhỏ nhất chưa phủ.
- Chứng minh tham lam nhanh: tìm `g` phần tử **đôi một không thể chung nhóm** (chặn dưới) và một cách chia `g` nhóm (chặn trên).
- Giá trị nhỏ (≤ 10⁵) ⇒ có thể bỏ sắp xếp bằng đếm phân phối.
- Bài cùng họ: 452 Minimum Number of Arrows to Burst Balloons, 1326 Minimum Number of Taps to Open to Water a Garden, 2366 Minimum Replacements to Sort the Array, 2856, 846 Hand of Straights, 2966 Divide Array Into Arrays With Max Difference.

## 15. Biến thể của bài

### 15.1 Biến thể A — Thêm giới hạn **kích thước nhóm** tối đa `cap`

Ràng buộc mới vẫn "đóng dưới tập con liên tiếp" (một đoạn con của nhóm hợp lệ vẫn hợp lệ) ⇒ tham lam "nhét càng nhiều càng tốt" vẫn tối ưu.

In [30]:
def with_cap(nums, k, cap):
    s = sorted(nums); ans, start, size = 1, s[0], 0
    for x in s:
        if x - start > k or size == cap:
            ans += 1; start = x; size = 0
        size += 1
    return ans

def with_cap_dp(nums, k, cap):
    s = sorted(nums); n = len(s); dp = [0] + [n + 1] * n
    for i in range(1, n + 1):
        for j in range(max(0, i - cap), i):
            if s[i - 1] - s[j] <= k: dp[i] = min(dp[i], dp[j] + 1)
    return dp[n]

rng = random.Random(151)
for _ in range(1500):
    a = [rng.randint(0, 20) for _ in range(rng.randint(1, 12))]; k = rng.randint(0, 6); cap = rng.randint(1, 5)
    assert with_cap(a, k, cap) == with_cap_dp(a, k, cap)
print(with_cap([1, 1, 1, 1, 2], 1, 2), "(5 phần tử, mỗi nhóm tối đa 2 -> 3)")

3 (5 phần tử, mỗi nhóm tối đa 2 -> 3)


### 15.2 Biến thể B — Chia thành **đoạn con liên tiếp** (giữ nguyên thứ tự)

Không được sắp xếp; tham lam "kéo dài đoạn hiện tại tới khi vi phạm" vẫn tối ưu (cùng lập luận: một đoạn con của đoạn hợp lệ vẫn hợp lệ). Chính là "Lỗi 2" ở trên.

In [31]:
def contiguous(nums, k):
    return wrong_no_sort(nums, k)

def contiguous_dp(nums, k):
    n = len(nums); dp = [0] + [n + 1] * n
    for i in range(1, n + 1):
        lo = hi = nums[i - 1]
        for j in range(i - 1, -1, -1):
            lo, hi = min(lo, nums[j]), max(hi, nums[j])
            if hi - lo > k: break
            dp[i] = min(dp[i], dp[j] + 1)
    return dp[n]

rng = random.Random(152)
for _ in range(1500):
    a = [rng.randint(0, 20) for _ in range(rng.randint(1, 12))]; k = rng.randint(0, 8)
    assert contiguous(a, k) == contiguous_dp(a, k)
print([contiguous(a, k) for a, k, _ in examples], "(so với dãy con:", [e[2] for e in examples], ")")

[4, 2, 3] (so với dãy con: [2, 2, 3] )


### 15.3 Biến thể C — Cho trước số nhóm tối đa `G`, tìm `k` **nhỏ nhất**

Đáp án `f(k)` không tăng theo `k` ⇒ **tìm nhị phân** trên `k ∈ [0, max − min]`.

In [32]:
def min_k_for_groups(nums, G):
    lo, hi = 0, max(nums) - min(nums)
    while lo < hi:
        mid = (lo + hi) // 2
        if Solution().partitionArray(nums, mid) <= G: hi = mid
        else: lo = mid + 1
    return lo

rng = random.Random(153)
for _ in range(300):
    a = [rng.randint(0, 40) for _ in range(rng.randint(1, 15))]; G = rng.randint(1, 5)
    k = min_k_for_groups(a, G)
    assert Solution().partitionArray(a, k) <= G and (k == 0 or Solution().partitionArray(a, k - 1) > G)
print(min_k_for_groups([3, 6, 1, 2, 5], 1), min_k_for_groups([3, 6, 1, 2, 5], 2), min_k_for_groups([3, 6, 1, 2, 5], 5))

5 2 0


### 15.4 Biến thể D — LC 2966: chia thành các nhóm **đúng 3 phần tử**, hiệu ≤ k (trả về nhóm hoặc `[]`)

Sắp xếp rồi cắt từng bộ 3 liên tiếp; nếu một bộ vi phạm thì không có cách nào (lập luận hoán đổi).

In [33]:
def divide_into_threes(nums, k):
    s = sorted(nums); out = []
    for i in range(0, len(s), 3):
        g = s[i:i + 3]
        if g[-1] - g[0] > k: return []
        out.append(g)
    return out

print(divide_into_threes([1, 3, 4, 8, 7, 9, 3, 5, 1], 2))
print(divide_into_threes([2, 4, 2, 2, 5, 2], 2))

[[1, 1, 3], [3, 4, 5], [7, 8, 9]]
[]


## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Vét cạn chia tập | định nghĩa | ~B(n) | O(n) | kiểm chứng |
| 1b. DP O(n²) | nhóm = đoạn của mảng sắp | O(n²) | O(n) | khi có thêm ràng buộc phức tạp |
| 2. Sắp xếp + quét | tham lam | O(n log n) | O(n) | **mặc định** |
| 3. `bisect` nhảy | nhảy cả nhóm | O(n log n + g log n) | O(n) | đã có mảng sắp, k lớn |
| 4. Đếm phân phối | đánh dấu giá trị | O(n + V) | O(V) | giá trị nhỏ |
| 5. Từ phía lớn | đối xứng | O(n log n) | O(n) | minh họa |

## 17. 5 bài tập vận dụng (từ dễ đến khó)

**Bài 1 (đoán output):** `partitionArray([1, 1, 1, 9], 0)` và `partitionArray([10, 20, 30], 15)`?

**Bài 2 (tính tay):** `nums = [16, 2, 9, 5, 11, 1, 3]`, `k = 3`. Liệt kê các nhóm tham lam và đáp án.

**Bài 3 (điền TODO):** Viết `bt3_todo(nums, k)` trả về **danh sách các nhóm** (mỗi nhóm là danh sách giá trị đã sắp) theo tham lam.

**Bài 4 (tự viết lại):** Viết `bt4_todo(nums, k)` cài cách **đếm phân phối** không nhìn code.

**Bài 5 (mini challenge):** Viết `bt5_todo(nums, k)` trả về số nhóm ít nhất khi điều kiện mỗi nhóm là `max ≤ 2 · min` (thay cho `max − min ≤ k`), với `nums[i] ≥ 1`.

In [34]:
def bt3_todo(nums, k):
    # TODO: các nhóm tham lam
    pass

def bt4_todo(nums, k):
    # TODO: đếm phân phối
    pass

def bt5_todo(nums):
    # TODO: điều kiện max <= 2 * min
    raise NotImplementedError

## 18. Gợi ý 3 mức cho bài tập

**Bài 1.** Mức 1: `k = 0` ⇒ số giá trị phân biệt. Mức 2: `{1, 9}` ⇒ 2. Mức 3: `[10, 20, 30]`, `k = 15`: nhóm `{10, 20}` (20 − 10 = 10), `30 − 10 = 20 > 15` ⇒ nhóm mới ⇒ `2`.

**Bài 2.** Mức 1: sắp xếp `[1, 2, 3, 5, 9, 11, 16]`. Mức 2: nhóm `{1, 2, 3}` (5 − 1 = 4 > 3 ⇒ cắt), `{5}` (9 − 5 = 4 > 3), `{9, 11}`. Mức 3: `{16}` ⇒ 4 nhóm.

**Bài 3.** Mức 1: sắp xếp. Mức 2: nhóm mới khi `x − nhóm[0] > k`. Mức 3: trả về danh sách nhóm.

**Bài 4.** Mức 1: `present = bytearray(V + 1)`. Mức 2: quét `v`, gặp giá trị có mặt thì `ans += 1`. Mức 3: nhảy `v += k + 1`.

**Bài 5.** Mức 1: điều kiện vẫn "đóng dưới đoạn con" của mảng sắp. Mức 2: tham lam: nhóm mới khi `x > 2 · start`. Mức 3: chặn dưới giống mục 10 (các điểm bắt đầu đôi một vi phạm).

## 19. Đáp án tham khảo cho bài tập

**Bài 1:** `2`, `2`. **Bài 2:** `[[1, 2, 3], [5], [9, 11], [16]]` ⇒ `4`.

In [35]:
assert Solution().partitionArray([1, 1, 1, 9], 0) == 2
assert Solution().partitionArray([10, 20, 30], 15) == 2

def bt3(nums, k):
    return greedy_groups(nums, k)

assert bt3([16, 2, 9, 5, 11, 1, 3], 3) == [[1, 2, 3], [5], [9, 11], [16]]

def bt4(nums, k):
    return counting(nums, k)

def bt5(nums):
    s = sorted(nums); ans, start = 1, s[0]
    for x in s:
        if x > 2 * start: ans += 1; start = x
    return ans

def bt5_brute(nums):
    n = len(nums); best = [n]; groups = []
    def go(i):
        if len(groups) >= best[0]: return
        if i == n: best[0] = len(groups); return
        x = nums[i]
        for g in range(len(groups)):
            lo, hi = groups[g]
            if max(hi, x) <= 2 * min(lo, x):
                groups[g] = (min(lo, x), max(hi, x)); go(i + 1); groups[g] = (lo, hi)
        groups.append((x, x)); go(i + 1); groups.pop()
    go(0); return best[0]

rng = random.Random(1989)
for _ in range(1000):
    a = [rng.randint(1, 30) for _ in range(rng.randint(1, 8))]; k = rng.randint(0, 6)
    g = bt3(a, k)
    assert len(g) == brute(a, k) and sorted(x for grp in g for x in grp) == sorted(a)
    assert bt4(a, k) == brute(a, k)
    assert bt5(a) == bt5_brute(a)
print(bt5([1, 2, 3, 4, 5, 9, 20]))
print("Đáp án bài tập: tất cả assert OK")

4
Đáp án bài tập: tất cả assert OK


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

```python
nums = sorted(nums)                 # dãy con => thứ tự không quan trọng
ans, start = 1, nums[0]
for x in nums:
    if x - start > k:               # so với MIN của nhóm, không phải phần tử liền trước
        ans += 1; start = x
return ans
```

**Checklist:**
- [ ] Đã sắp xếp (bài là **dãy con**, không phải đoạn con).
- [ ] So với `start` của nhóm, không so với phần tử trước.
- [ ] Dùng `> k` (hiệu bằng `k` vẫn hợp lệ).
- [ ] `ans` bắt đầu từ 1 (nhóm đầu tiên).